# Fine-tuned Qwen3-4B (`checkpoint-0792`) on the test splits (Milestone 3, test run #5)

Runs the chosen checkpoint (`docs/milestone-3/VALIDATION_LOG.md`) on every declared test split **once**, under the frozen protocol (`docs/milestone-2/EVAL_PROTOCOL.md`, "Test-split evaluation", and Amendment 1):

| split | cases |
|---|---|
| tier 1 `test_in_distribution` (splits v1, dataset v2) | 560 |
| tier 1 `test_held_out` | 960 |
| tier 2 `test_in_distribution` (splits v2, dataset v3) | 256 |
| tier 2 `test_held_out` | 400 |
| live test v1 | 300 |

Settings: the adapter merged into the fp16 base, the model's chat template, greedy decoding, 128 new tokens, prompt version 2, one generation at a time on **one visible T4** per process. `python -m packages.support_evaluator.checkpoint_selection test` checks the checkpoint against its logged `state.json` sha256 and against the selection run's `choice.json` (itself checked against its logged sha256).

**This notebook prints no results**, only whether each split is complete and the call health counts. The run is logged in `docs/milestone-2/TEST_RUN_LOG.md` before any result is read. A split that stops is resumed only to complete it, never repeated to change it.

**On Kaggle (T4 x2):** two processes, one T4 each. GPU 0 runs tier 1 `test_held_out`, then tier 2 `test_in_distribution`; GPU 1 runs tier 2 `test_held_out`, then tier 1 `test_in_distribution`, then live. `schedule.jsonl` records each split's GPU, start and end.

**On Colab (one T4):**
- **Order:** the splits run one after another. First, the fp16 smoke test runs on the new platform: the same 10 seeded validation cases per tier as the Kaggle full validation run, so their episodes can be compared across platforms. If it fails, no test split starts.
- **Writing:** the runner writes to the local disk (`/content/qlora-test`). Every minute, and at the end, a mirror on Google Drive (`MyDrive/hetren-m3/qlora-test`) is updated with complete lines only, rows before calls.
- **After a disconnect:** run all again. The notebook restores the mirror, and scored cases are not repeated. Calls of a case that was not yet scored are marked `discarded`, the runner's convention for unscored calls, and that case runs again.
- **Environment:** `environment.json` records the platform, GPU, driver, CUDA and cuDNN builds and `pip freeze`. The first cell prints every library version that differs from the Kaggle runs.

**Setup, Kaggle:**

- Accelerator *GPU T4 x2*, Internet *on*, Environment *Pin to original environment*.
- Inputs:
  - the evaluation bundle dataset;
  - the session 2 retry training output, which holds `checkpoint-0792`;
  - the checkpoint-selection output (`selection-run-1`), which holds `qlora-selection/choice.json`.
- Set `BUNDLE_COMMIT`, then *Save Version → Save & Run All*.

**Setup, Colab:**

- *Runtime → Change runtime type → T4 GPU*.
- On Google Drive, under `MyDrive/hetren-m3/`, put:
  - the bundle `.tar.gz`, and no other bundle;
  - `checkpoints/checkpoint-0792/`: its `state.json` and the `adapter/` folder;
  - `qlora-selection/choice.json`.
- Set `PLATFORM = "colab"` and `BUNDLE_COMMIT`, then *Runtime → Run all*. Keep the tab open.
- After a disconnect: reconnect and *Run all* again.

**If the session cannot run every split:** set `SPLITS` to the splits for this session. In the next session, attach this notebook's earlier output: complete and partial splits are carried over, and partial ones are completed.

In [ ]:
PLATFORM = "kaggle"  # "kaggle" or "colab"
BUNDLE_COMMIT = ""  # the evaluation bundle's commit (full sha); the notebook stops if the bundle found is another
CHOSEN = 792
STATE_SHA256 = "d6230ff6e1780ac5286f0accc0a45298dd8db49fc11ed6fdb1330d98189276d5"  # checkpoint-0792 (validation log)
CHOICE_SHA256 = "d41a7ef595e8882ff0dbb367e62b03b87c8f1006bb60da09811fb19f11992332"  # selection-run-1 choice.json
SPLITS = ["tier1-test_held_out", "tier2-test_held_out"]  # session B (session A ran the other three)
# Two T4s: about 2.3 h of expected work on GPU 0 and 2.7 h on GPU 1, longest first. One T4: all in this order.
ASSIGN = {0: ["tier1-test_held_out", "tier2-test_in_distribution"],
          1: ["tier2-test_held_out", "tier1-test_in_distribution", "live"]}
SIZES = {"tier1-test_in_distribution": 560, "tier1-test_held_out": 960, "tier2-test_in_distribution": 256,
         "tier2-test_held_out": 400, "live": 300}
DRIVE = "/content/drive/MyDrive/hetren-m3"  # Colab only

import glob, hashlib, json, os, shutil, subprocess, sys, tarfile, threading, time

if PLATFORM == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    INPUTS, WORK, OUT = DRIVE, "/content/hetren", "/content/qlora-test"  # mirrored to MIRROR on Drive
    MIRROR = f"{DRIVE}/qlora-test"
else:
    INPUTS, WORK, OUT = "/kaggle/input", "/kaggle/working/hetren", "/kaggle/working/qlora-test"


def earlier_session(path):  # a copy of the code inside an attached earlier notebook output, not the bundle itself
    return bool(glob.glob(f"{path.split('/hetren/')[0]}/qlora-*")) if "/hetren/" in path else False


if not os.path.exists(WORK):
    unpacked = [p for p in glob.glob(f"{INPUTS}/**/BUNDLE.json", recursive=True) if not earlier_session(p)]
    packed = [p for p in glob.glob(f"{INPUTS}/**/hetren-*bundle.tar.gz", recursive=True) if not earlier_session(p)]

    if len(unpacked) > 1 or (not unpacked and len(packed) != 1):
        raise SystemExit(f"STOP: attach exactly one bundle; found {unpacked or packed}")

    if unpacked:  # Kaggle may unpack archives on upload
        shutil.copytree(os.path.dirname(unpacked[0]), WORK)
    else:
        os.makedirs(WORK)
        tarfile.open(packed[0]).extractall(WORK)

    print("bundle from", unpacked[0] if unpacked else packed[0])

COMMIT = json.load(open(f"{WORK}/BUNDLE.json"))["commit"]
print("bundle commit:", COMMIT)

if BUNDLE_COMMIT and COMMIT != BUNDLE_COMMIT.strip():
    raise SystemExit(f"STOP: the bundle is commit {COMMIT}, not BUNDLE_COMMIT {BUNDLE_COMMIT!r}")

os.chdir(WORK)

In [ ]:
# The evaluation libraries, as in the selection and validation runs; torch (and Python) are the platform's.
!pip install -q jsonschema==4.26.0 openai==3.42.0 openai-agents==0.24.0 tiktoken==0.14.0 transformers==4.56.2 accelerate==1.10.1 peft==0.17.1 tokenizers==0.22.2
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv
GPUS = [line.split(",")[1].strip() for line in subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name", "--format=csv,noheader"], capture_output=True, text=True).stdout.splitlines()]

if not GPUS or any("T4" not in name for name in GPUS):
    raise SystemExit(f"STOP: the GPUs are {GPUS}; the protocol evaluates on T4s")

print(len(GPUS), "T4s; each evaluation process sees one")

# The environment, recorded with the outputs; differences from the Kaggle runs are printed before anything runs.
KAGGLE = {"python": "3.13.16", "torch": "2.11.0+cu130", "transformers": "4.56.2", "accelerate": "1.10.1",
          "tokenizers": "0.22.2", "peft": "0.17.1", "jsonschema": "4.26.0", "openai-agents": "0.24.0", "tiktoken": "0.14.0"}
probe = ("import json, torch; from packages.hetren_baselines.local_model import library_versions; "
         "print(json.dumps({'libraries': library_versions(), 'cuda': torch.version.cuda, "
         "'cudnn': torch.backends.cudnn.version()}))")
ENVIRONMENT = json.loads(subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True, check=True).stdout)
ENVIRONMENT.update(platform=PLATFORM, gpus=GPUS, driver=subprocess.run(
    ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"], capture_output=True, text=True).stdout.split(),
    pip_freeze=subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout.splitlines())
DIFFERENCES = {k: (v, ENVIRONMENT["libraries"].get(k)) for k, v in KAGGLE.items() if ENVIRONMENT["libraries"].get(k) != v}
print("CUDA", ENVIRONMENT["cuda"], "cuDNN", ENVIRONMENT["cudnn"], "driver", ENVIRONMENT["driver"])
print("differences from the Kaggle runs (Kaggle, here):", DIFFERENCES or "none")

## The chosen checkpoint and the selection run's choice, checked by hash; the base model downloaded once

In [ ]:
def find(pattern, digest):
    found = sorted(glob.glob(f"{INPUTS}/**/{pattern}", recursive=True))
    matching = [p for p in found if hashlib.sha256(open(p, "rb").read()).hexdigest() == digest]

    if not matching:
        raise SystemExit(f"STOP: no {pattern} with sha256 {digest} among {found}")

    return matching[0]


CHECKPOINT = os.path.dirname(find(f"checkpoint-{CHOSEN:04d}/state.json", STATE_SHA256))
CHOICE = find("qlora-selection/choice.json", CHOICE_SHA256)

if json.load(open(CHOICE))["chosen"] != f"checkpoint-{CHOSEN:04d}":
    raise SystemExit(f"STOP: choice.json chose {json.load(open(CHOICE))['chosen']}, not checkpoint-{CHOSEN:04d}")

print("checkpoint:", CHECKPOINT)
print("choice:", CHOICE)

# Splits of an earlier session (attached output of this notebook) are carried over, complete or partial.
for earlier in glob.glob(f"{INPUTS}/**/qlora-test/*/metadata.json", recursive=True):
    target = f"{OUT}/{os.path.basename(os.path.dirname(earlier))}"

    if PLATFORM == "kaggle" and not os.path.exists(target):
        shutil.copytree(os.path.dirname(earlier), target)
        print("carried over", os.path.dirname(earlier))


def mark_unscored_calls(directory):
    # Calls of a case with no row yet (cut off by a disconnect) are marked discarded; the case runs again.
    for calls_path in glob.glob(f"{directory}/**/calls.jsonl", recursive=True):
        folder = os.path.dirname(calls_path)
        scored = {json.loads(line)["case_id"] for rows in glob.glob(f"{folder}/rows-*.jsonl") for line in open(rows)}
        calls = [json.loads(line) for line in open(calls_path)]
        unscored = [c for c in calls if c["case_id"] not in scored and not c.get("discarded")]

        if unscored:
            with open(calls_path, "w") as handle:
                handle.writelines(json.dumps({**c, "discarded": c.get("discarded") or c["case_id"] not in scored}) + "\n"
                                  for c in calls)

            print(f"{folder}: {len(unscored)} calls of unscored cases marked discarded")


if PLATFORM == "colab" and os.path.exists(MIRROR) and not os.path.exists(OUT):  # resume after a disconnect
    shutil.copytree(MIRROR, OUT)
    mark_unscored_calls(OUT)
    print("restored from", MIRROR)

os.makedirs(OUT, exist_ok=True)
json.dump(ENVIRONMENT, open(f"{OUT}/environment-{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}.json", "w"), indent=1)

subprocess.run([sys.executable, "-c", "from huggingface_hub import snapshot_download; "
                "from packages.hetren_baselines.local_model import QWEN_MODEL, QWEN_REVISION; "
                "print(snapshot_download(QWEN_MODEL, revision=QWEN_REVISION))"], check=True)

## Test runs (each split once; about 3 hours on two T4s)

The per-split logs (`qlora-test/logs/<split>.log`) list each case's outcome; they are part of the results and are read only after the run is logged.

In [ ]:
os.makedirs(f"{OUT}/logs", exist_ok=True)
STOP_MIRROR = threading.Event()


def mirror_once():
    # Complete lines only, rows before calls: the mirror never holds a half line or a scored case without its calls.
    files = sorted((p for p in glob.glob(f"{OUT}/**/*", recursive=True) if os.path.isfile(p)),
                   key=lambda p: (os.path.dirname(p), os.path.basename(p) == "calls.jsonl", p))

    for path in files:
        target = f"{MIRROR}/{os.path.relpath(path, OUT)}"
        os.makedirs(os.path.dirname(target), exist_ok=True)
        data = open(path, "rb").read()

        if path.endswith(".jsonl"):
            data = data[:data.rfind(b"\n") + 1]

        with open(target + ".part", "wb") as handle:
            handle.write(data)

        os.replace(target + ".part", target)


def mirror_loop():
    while not STOP_MIRROR.wait(60):
        mirror_once()


if PLATFORM == "colab":
    threading.Thread(target=mirror_loop, daemon=True).start()
    smoke = [sys.executable, "-m", "packages.support_evaluator.checkpoint_selection", "smoke", "--checkpoint", CHECKPOINT,
             "--state-sha256", STATE_SHA256, "--choice", CHOICE, "--choice-sha256", CHOICE_SHA256,
             "--out", f"{OUT}/platform-smoke"]

    with open(f"{OUT}/logs/platform-smoke.log", "a") as log:  # the fp16 smoke test on the new platform
        code = subprocess.run(smoke, stdout=log, stderr=subprocess.STDOUT, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"}).returncode

    mirror_once()

    if code != 0:
        raise SystemExit(f"STOP: the fp16 smoke test on this platform exited with {code}; no test split runs")

    print("fp16 smoke test passed on this platform:",
          {t: json.load(open(f"{OUT}/platform-smoke/smoke-tier{t}-validation/smoke_report.json"))["fp16_ok"] for t in (1, 2)})


def complete(split):
    rows = f"{OUT}/{split}/rows-{split}.jsonl"
    return os.path.exists(rows) and sum(1 for _ in open(rows)) == SIZES[split]


pending = [s for s in SPLITS if not complete(s)]
queues = ({gpu: [s for s in ASSIGN[gpu] if s in pending] for gpu in ASSIGN} if len(GPUS) == 2 else {0: pending})
EXITS = {}
SCHEDULE = f"{OUT}/schedule.jsonl"  # which split ran on which GPU and when (for the test-run log)
utc = lambda: time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
lock = threading.Lock()


def worker(gpu, splits):
    for split in splits:
        command = [sys.executable, "-m", "packages.support_evaluator.checkpoint_selection", "test",
                   "--checkpoint", CHECKPOINT, "--state-sha256", STATE_SHA256, "--choice", CHOICE,
                   "--choice-sha256", CHOICE_SHA256, "--split", split, "--out", f"{OUT}/{split}"]

        with open(f"{OUT}/logs/{split}.log", "a") as log:
            started, began = time.time(), utc()
            EXITS[split] = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT,
                                          env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}).returncode

        with lock, open(SCHEDULE, "a") as schedule:
            schedule.write(json.dumps({"checkpoint": f"checkpoint-{CHOSEN:04d}", "split": split, "gpu": gpu,
                                       "gpu_name": GPUS[gpu], "started": began, "finished": utc(),
                                       "exit": EXITS[split], "platform": PLATFORM}) + "\n")

        print(f"{split} on GPU {gpu}: exit {EXITS[split]} after {(time.time() - started) / 3600:.2f} h", flush=True)


threads = [threading.Thread(target=worker, args=(gpu, splits)) for gpu, splits in queues.items()]

for thread in threads:
    thread.start()

for thread in threads:
    thread.join()

for split in SPLITS:  # completeness and call health only: no result is printed
    if complete(split):
        calls = [json.loads(line) for line in open(f"{OUT}/{split}/calls.jsonl")]
        print(f"{split}: complete, {SIZES[split]} cases; calls {len(calls)}, non-finite {sum(not c['finite'] for c in calls)}, "
              f"empty {sum(not c['text'].strip() for c in calls)}, at limit {sum(c['at_limit'] for c in calls)}")
    else:
        print(f"{split}: NOT complete (exit {EXITS.get(split)})")

if PLATFORM == "colab":
    STOP_MIRROR.set()
    mirror_once()
    print("mirrored to", MIRROR)

## Outputs

In [ ]:
archive = ("/kaggle/working" if PLATFORM == "kaggle" else DRIVE) + "/qlora-test.tar.gz"

with tarfile.open(archive, "w:gz") as results:
    results.add(OUT, arcname="qlora-test")

print("archive", archive, hashlib.sha256(open(archive, "rb").read()).hexdigest())

if any(code != 0 for code in EXITS.values()) or not all(complete(s) for s in SPLITS):
    raise SystemExit(f"incomplete: {EXITS}; log this run before resuming")